In [ ]:
import requests
#API_KEY = Your key here
headers = {"X-API-Key": API_KEY}

def fetch_locations(country_ids: dict) -> dict:
    """Takes {country_name: country_id} and returns {country_name: list_of_locations}."""
    results = {}
    for country, country_id in country_ids.items():
        all_locations = []
        page = 1
        while True:
            response = requests.get(
                "https://api.openaq.org/v3/locations",
                params={"countries_id": country_id, "limit": 1000, "page": page,},# "providers_id": 119},
                headers=headers
            )
            data = response.json()
            page_results = data["results"]
            if not page_results:
                break
            all_locations.extend(page_results)
            print(f"{country} page {page}: fetched {len(page_results)} | total: {len(all_locations)}")
            page += 1
        results[country] = all_locations
    return results


In [ ]:
codes={"Chile": 3,"Germany": 50, "France": 22, "Australia": 177, "Brazil": 45}


In [ ]:
locations = fetch_locations(codes)

In [ ]:
def filter_monitors(locations: dict) -> dict:
    return {country: [loc for loc in locs if loc["isMonitor"]] for country, locs in locations.items()}

monitor_locations = filter_monitors(locations)

location_ids = {country: [loc["id"] for loc in locs] for country, locs in monitor_locations.items()}


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
from tqdm import tqdm

BUCKET = "openaq-data-archive"
YEARS = range(2022, 2026)
OUT_DIR = "./openaq_raw2"
os.makedirs(OUT_DIR, exist_ok=True)
# Retry with exponential backoff to handle S3 throttling
s3 = boto3.client("s3", config=Config(
    signature_version=UNSIGNED,
    max_pool_connections=64,
    retries={"max_attempts": 10, "mode": "adaptive"},
))

def download_location_year(args):
    country, loc_id, year = args
    prefix = f"records/csv.gz/locationid={loc_id}/year={year}/"
    paginator = s3.get_paginator("list_objects_v2")

    downloaded = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            local_path = os.path.join(OUT_DIR, country, key)
            if os.path.exists(local_path):
                continue
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            s3.download_file(BUCKET, key, local_path)
            downloaded += 1
    return country, loc_id, year, downloaded

tasks = [
    (country, loc_id, year)
    for country, ids in location_ids.items()
    for loc_id in ids
    for year in YEARS
]

with ThreadPoolExecutor(max_workers=64) as executor:
    futures = {executor.submit(download_location_year, t): t for t in tasks}
    with tqdm(total=len(tasks), desc="Downloading", unit="task") as pbar:
        for future in as_completed(futures):
            country, loc_id, year, n = future.result()
            pbar.update(1)